### Hierarichal agents with RAG

In [ ]:
import operator
import os
import re
from typing import Annotated, List, Literal

import wikipedia
from ddgs import DDGS
from dotenv import load_dotenv
from IPython.display import Image, display
from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send

load_dotenv()  

In [26]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="qwen3:14b", num_ctx=8192, reasoning=False)

In [16]:
def duck_duck_search(query: str):
    results = DDGS().text(query, max_results=5)
    return results


In [19]:
alo = duck_duck_search("who is cristiano ronaldo ?")


In [20]:
alo

[{'title': 'Cristiano Ronaldo - WikipediaCristiano Ronaldo: Biography, Soccer Player, Al-Nassr, 2026 ...Cristiano Ronaldo | 2026 FIFA World Cup, Goals, Portugal ...Cristiano Ronaldo Facts | BritannicaCristiano Ronaldo - ForbesCristiano Ronaldo: Stats, quotes, highlights, trivia and quizCareer of Cristiano Ronaldo - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Cristiano_Ronaldo',
  'body': "Cristiano Ronaldo dos Santos Aveiro[C] (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains both the Saudi Pro League club Al-Nassr and the Portugal national team. Nicknamed CR7, [D] he is widely regarded as one of the greatest footballers of all time. Jun 24, 2026 · Portuguese soccer superstar Cristiano Ronaldo is one of the greatest athletes in the sport’s history, winning the Ballon d’Or five times and becoming the first player to score 900 career goals. Cristiano Ronaldo was born on February 5, 1985, in Funchal, Madeira, Portugal. What was

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

@tool 
def Web_scraper(urls:List[str]) ->str:
    """Use this tool to scrap webpages from the given urls"""
    docs = WebBaseLoader(urls).load()

    return "\n\n".join(
        [
            f'<Document name="{doc.metadata.get("title", "")}">\n {doc.page_content}\n</Document>'
            for doc in docs
        ]

    )


: 